# Time Series Clustering with Pretrained Models

## Assignment (f): Clustering Time Series Data using Pretrained Models

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. Introduction
2. Dataset Preparation
3. Time Series Feature Extraction
4. Clustering with TSlearn
5. DTW-based Clustering
6. Stock Market Clustering
7. Evaluation Metrics
8. Conclusion

In [ ]:
!pip install tslearn numpy pandas matplotlib seaborn scikit-learn yfinance tsfresh -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Base libraries imported!")

In [ ]:
from tslearn.clustering import TimeSeriesKMeans, KernelKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.metrics import dtw, soft_dtw
from tslearn.datasets import CachedDatasets
from tslearn.barycenters import dtw_barycenter_averaging
print("TSlearn imported successfully!")

## 1. Load UCR Time Series Dataset

In [ ]:
X_train, y_train, X_test, y_test = CachedDatasets().load_dataset("Trace")
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])
print(f"Dataset shape: {X_all.shape}")
print(f"Number of classes: {len(np.unique(y_all))}")
print(f"Time series length: {X_all.shape[1]}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, cls in enumerate(np.unique(y_all)[:4]):
    ax = axes[i // 2, i % 2]
    class_samples = X_all[y_all == cls][:5]
    for sample in class_samples:
        ax.plot(sample.ravel(), alpha=0.7)
    ax.set_title(f'Class {int(cls)}')
    ax.set_xlabel('Time'); ax.set_ylabel('Value')
plt.suptitle('Sample Time Series by Class', fontsize=14)
plt.tight_layout(); plt.show()

## 2. Time Series Normalization

In [ ]:
scaler = TimeSeriesScalerMeanVariance()
X_scaled = scaler.fit_transform(X_all)
print(f"Scaled data shape: {X_scaled.shape}")

## 3. K-Means with Euclidean Distance

In [ ]:
n_clusters = len(np.unique(y_all))
kmeans_euclidean = TimeSeriesKMeans(n_clusters=n_clusters, metric="euclidean", max_iter=50, random_state=42)
labels_euclidean = kmeans_euclidean.fit_predict(X_scaled)
print(f"Euclidean K-Means:")
print(f"  Silhouette: {silhouette_score(X_scaled.reshape(len(X_scaled), -1), labels_euclidean):.4f}")
print(f"  ARI: {adjusted_rand_score(y_all, labels_euclidean):.4f}")

## 4. K-Means with DTW (Dynamic Time Warping)

In [ ]:
kmeans_dtw = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", max_iter=30, random_state=42)
labels_dtw = kmeans_dtw.fit_predict(X_scaled)
print(f"DTW K-Means:")
print(f"  ARI: {adjusted_rand_score(y_all, labels_dtw):.4f}")
print(f"  NMI: {normalized_mutual_info_score(y_all, labels_dtw):.4f}")

In [ ]:
fig, axes = plt.subplots(1, n_clusters, figsize=(16, 4))
for i in range(n_clusters):
    ax = axes[i]
    cluster_samples = X_scaled[labels_dtw == i][:10]
    for sample in cluster_samples:
        ax.plot(sample.ravel(), alpha=0.3, color='gray')
    ax.plot(kmeans_dtw.cluster_centers_[i].ravel(), 'r-', linewidth=2, label='Centroid')
    ax.set_title(f'Cluster {i} (n={np.sum(labels_dtw == i)})')
    ax.legend()
plt.suptitle('DTW K-Means Cluster Centers', fontsize=14)
plt.tight_layout(); plt.show()

## 5. Soft-DTW K-Means

In [ ]:
kmeans_softdtw = TimeSeriesKMeans(n_clusters=n_clusters, metric="softdtw", metric_params={"gamma": 0.1}, max_iter=30, random_state=42)
labels_softdtw = kmeans_softdtw.fit_predict(X_scaled)
print(f"Soft-DTW K-Means:")
print(f"  ARI: {adjusted_rand_score(y_all, labels_softdtw):.4f}")
print(f"  NMI: {normalized_mutual_info_score(y_all, labels_softdtw):.4f}")

## 6. Generate Synthetic Time Series Data

In [ ]:
def generate_synthetic_ts(n_samples=50, length=100):
    t = np.linspace(0, 4*np.pi, length)
    ts_data, labels = [], []
    # Class 0: Sine waves
    for _ in range(n_samples):
        ts_data.append(np.sin(t + np.random.uniform(0, 0.5)) + np.random.normal(0, 0.1, length))
        labels.append(0)
    # Class 1: Cosine waves
    for _ in range(n_samples):
        ts_data.append(np.cos(t + np.random.uniform(0, 0.5)) + np.random.normal(0, 0.1, length))
        labels.append(1)
    # Class 2: Sawtooth-like
    for _ in range(n_samples):
        ts_data.append(np.abs(np.sin(t * 2)) + np.random.normal(0, 0.1, length))
        labels.append(2)
    # Class 3: Random walk
    for _ in range(n_samples):
        ts_data.append(np.cumsum(np.random.normal(0, 0.1, length)))
        labels.append(3)
    return np.array(ts_data), np.array(labels)

X_synth, y_synth = generate_synthetic_ts()
X_synth = X_synth.reshape(X_synth.shape[0], X_synth.shape[1], 1)
print(f"Synthetic data: {X_synth.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
class_names = ['Sine', 'Cosine', 'Sawtooth', 'Random Walk']
for i in range(4):
    ax = axes[i // 2, i % 2]
    samples = X_synth[y_synth == i][:5]
    for s in samples:
        ax.plot(s.ravel(), alpha=0.7)
    ax.set_title(f'Class {i}: {class_names[i]}')
plt.tight_layout(); plt.show()

In [ ]:
X_synth_scaled = TimeSeriesScalerMeanVariance().fit_transform(X_synth)
kmeans_synth = TimeSeriesKMeans(n_clusters=4, metric="dtw", max_iter=30, random_state=42)
labels_synth = kmeans_synth.fit_predict(X_synth_scaled)
print(f"Synthetic Data Clustering:")
print(f"  ARI: {adjusted_rand_score(y_synth, labels_synth):.4f}")
print(f"  NMI: {normalized_mutual_info_score(y_synth, labels_synth):.4f}")

## 7. Stock Market Time Series Clustering

In [ ]:
import yfinance as yf
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM', 'BAC', 'WFC', 'GS', 'XOM', 'CVX', 'COP', 'SLB', 'JNJ', 'PFE', 'UNH', 'MRK']
stock_data = {}
for ticker in tickers:
    try:
        data = yf.download(ticker, start='2023-01-01', end='2024-01-01', progress=False)
        if len(data) > 200:
            stock_data[ticker] = data['Close'].values[:200]
    except: pass
print(f"Downloaded {len(stock_data)} stocks")

In [ ]:
if len(stock_data) > 5:
    stock_names = list(stock_data.keys())
    X_stocks = np.array([stock_data[t] for t in stock_names])
    # Normalize to returns
    X_returns = np.diff(X_stocks, axis=1) / X_stocks[:, :-1] * 100
    X_stocks_scaled = StandardScaler().fit_transform(X_returns)
    X_stocks_ts = X_stocks_scaled.reshape(X_stocks_scaled.shape[0], X_stocks_scaled.shape[1], 1)
    
    kmeans_stocks = TimeSeriesKMeans(n_clusters=4, metric="dtw", max_iter=30, random_state=42)
    stock_labels = kmeans_stocks.fit_predict(X_stocks_ts)
    
    print("\nStock Clustering Results:")
    for i in range(4):
        cluster_stocks = [stock_names[j] for j in range(len(stock_names)) if stock_labels[j] == i]
        print(f"  Cluster {i}: {cluster_stocks}")

In [ ]:
if len(stock_data) > 5:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for i in range(4):
        ax = axes[i // 2, i % 2]
        cluster_idx = np.where(stock_labels == i)[0]
        for idx in cluster_idx:
            ax.plot(X_stocks_scaled[idx], alpha=0.7, label=stock_names[idx])
        ax.set_title(f'Cluster {i}')
        ax.legend(fontsize=8)
    plt.suptitle('Stock Returns Clustering (DTW)', fontsize=14)
    plt.tight_layout(); plt.show()

## 8. Feature-based Clustering with tsfresh

In [ ]:
def extract_ts_features(X):
    features = []
    for ts in X:
        ts = ts.ravel()
        feat = [np.mean(ts), np.std(ts), np.min(ts), np.max(ts), np.median(ts),
                np.percentile(ts, 25), np.percentile(ts, 75), np.sum(np.abs(np.diff(ts))),
                np.argmax(ts), np.argmin(ts), len(np.where(np.diff(np.sign(ts)))[0])]
        features.append(feat)
    return np.array(features)

X_features = extract_ts_features(X_scaled)
X_features_scaled = StandardScaler().fit_transform(X_features)
kmeans_features = KMeans(n_clusters=n_clusters, random_state=42)
labels_features = kmeans_features.fit_predict(X_features_scaled)
print(f"Feature-based Clustering:")
print(f"  ARI: {adjusted_rand_score(y_all, labels_features):.4f}")
print(f"  NMI: {normalized_mutual_info_score(y_all, labels_features):.4f}")

## 9. Model Comparison

In [ ]:
results = pd.DataFrame([
    {'Method': 'Euclidean K-Means', 'ARI': adjusted_rand_score(y_all, labels_euclidean), 'NMI': normalized_mutual_info_score(y_all, labels_euclidean)},
    {'Method': 'DTW K-Means', 'ARI': adjusted_rand_score(y_all, labels_dtw), 'NMI': normalized_mutual_info_score(y_all, labels_dtw)},
    {'Method': 'Soft-DTW K-Means', 'ARI': adjusted_rand_score(y_all, labels_softdtw), 'NMI': normalized_mutual_info_score(y_all, labels_softdtw)},
    {'Method': 'Feature-based', 'ARI': adjusted_rand_score(y_all, labels_features), 'NMI': normalized_mutual_info_score(y_all, labels_features)}
])
print("\nMethod Comparison:")
print(results.sort_values('ARI', ascending=False).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
ax.bar(x - 0.2, results['ARI'], 0.4, label='ARI', color='steelblue')
ax.bar(x + 0.2, results['NMI'], 0.4, label='NMI', color='coral')
ax.set_xticks(x); ax.set_xticklabels(results['Method'], rotation=15)
ax.set_ylabel('Score'); ax.set_title('Time Series Clustering Methods Comparison')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 10. Conclusion

### Key Findings:
- **DTW-based methods** generally outperform Euclidean distance for time series with temporal shifts
- **Soft-DTW** provides differentiable alternative to DTW
- **Feature extraction** can be effective when combined with traditional clustering

### Applications:
- Stock market analysis
- Sensor data clustering
- ECG/medical signal analysis
- Speech pattern recognition

In [ ]:
print("="*60)
print("TIME SERIES CLUSTERING - COMPLETE")
print("="*60)
print("\n✓ UCR dataset clustering")
print("✓ DTW and Soft-DTW K-Means")
print("✓ Synthetic time series generation")
print("✓ Stock market clustering")
print("✓ Feature-based clustering")
print("✓ Comprehensive evaluation")